# Biodiversity in National Parks
## Statistical Analysis of Species Conservation Status

**Data source:** National Park Service (NPS)  
**Author:** Óscar Jover Arrate  
**Tools:** Python · pandas · seaborn · scipy · numpy

---

### Project Overview

This notebook analyzes biodiversity data from the National Park Service, 
focusing on the conservation status of species across four major U.S. national parks.

The analysis addresses three core questions:
1. What is the distribution of conservation status across species categories?
2. Are certain taxonomic groups statistically more likely to be endangered?
3. Which species are most frequently observed, and does observation frequency 
   correlate with conservation concern?

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import chi2_contingency

# ── Plot configuration (dark mode) ────────────────────────────────────────────
plt.style.use('dark_background')

PALETTE = {
    'primary'   : '#4FC3F7',   # light blue
    'accent'    : '#FF8A65',   # warm orange
    'warn'      : '#FFD54F',   # amber
    'danger'    : '#EF5350',   # red
    'neutral'   : '#90A4AE',   # grey-blue
}

sns.set_theme(
    style='darkgrid',
    rc={
        'axes.facecolor'  : '#1E1E2E',
        'figure.facecolor': '#12121C',
        'grid.color'      : '#2E2E4E',
        'text.color'      : '#E0E0E0',
        'axes.labelcolor' : '#E0E0E0',
        'xtick.color'     : '#E0E0E0',
        'ytick.color'     : '#E0E0E0',
    }
)

pd.set_option('display.max_columns', 10)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Setup complete.")

## 1. Data Loading & Initial Exploration

In [ ]:
# Loads both data files in a DataFrame
observations = pd.read_csv('observations.csv')
species = pd.read_csv('species_info.csv')

### Observations File

In [ ]:
# Shows first five rows of the observations file
observations.head()

In [ ]:
# Shape of the table
print(f"Shape: {observations.shape[0]:,} rows × {observations.shape[1]} columns")
# Number of unique species
print(f"Unique species: {observations['scientific_name'].nunique():,}")
# Parks covered in the file
print(f"Parks covered:  {observations['park_name'].nunique()}")
print()
# Data types for each column
print('Data Types of Columns:')
print(observations.dtypes)
print()
# Missing values per column
print("Missing values:")
print(observations.isnull().sum())

### Species Info File

In [ ]:
# Shows first five rows of the species file
species.head()

In [ ]:
# Shape of the table
print(f"Shape: {species.shape[0]:,} rows × {species.shape[1]} columns")
print()
# Number of unique species
print(f"Categories of species: {species['category'].nunique():,}")
print('Categories:')
print(species['category'].value_counts().to_string())
print()
# Data types for each column
print('Data Types of Columns:')
print(species.dtypes)
print()
# Missing values per column
print("Missing values:")
print(species.isnull().sum())
# Unique conservation statuses
print('Unique Conservation Statuses:')
print(species.conservation_status.value_counts(dropna = False))
# Percentage of Null Values
consv_stat_null = species.conservation_status.isnull().sum()
prop_null = (consv_stat_null/len(species))*100
print(f'Percentage of Null Values: {prop_null:.2f}%')

### Key Observations

- `observations.csv`: **23,296 records** across **4 national parks** and **5,541 unique 
  species**. No missing values — the dataset is complete and ready to use.

- `species_info.csv`: **5,824 species** across **7 taxonomic categories** 
  (Vascular Plant, Bird, Nonvascular Plant, Mammal, Fish, Amphibian, Reptile).

- **96.7% of species have no recorded conservation status** (5,633 out of 5,824). 
  This is not missing data in the traditional sense — it means those species have no 
  active conservation concern flagged by the NPS. They will be labeled as 
  *No Concern* throughout this analysis.

- Of the **191 species with an active status**, the breakdown is:
  - Species of Concern: 161
  - Endangered: 16
  - Threatened: 10
  - In Recovery: 4

This distribution is heavily imbalanced, which will be a key consideration 
in the statistical testing performed in Section 4.

## 2. Conservation Status Analysis

In [ ]:
# New Column where NaN values are replaced by 'No Concern'
species['conservation_status_clean'] = species.conservation_status.fillna('No Concern')
# New Column 'is_protected', where True is for values different from 'No Concern'
species['is_protected'] = species.conservation_status_clean != 'No Concern'
print(species.is_protected.value_counts())

In [ ]:
# Set Table of categories and number of species from each category
species_per_category = species.category.value_counts(ascending= True).reset_index()
species_per_category.columns = ['category', 'num_species']

# Bar plot category vs number of species of each category
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data = species_per_category, x = 'num_species', y = 'category', color = PALETTE['primary'], ax = ax)
ax.set_title('Number of Species per Category')
ax.set_ylabel('Category')
ax.set_xlabel('Number of Species')
ax.set_xlim(xmin = 0, xmax =4750)
for bar in ax.patches:
    width = bar.get_width()
    ax.text(width + 50, bar.get_y() + bar.get_height()/2,
            f'{width:.0f}', va='center', fontsize=9,
            color='#E0E0E0')
plt.tight_layout()


# Saves the image in the images folder
plt.savefig('images/Number of Species per Category.png', dpi=150, bbox_inches='tight')

plt.show()
plt.close()

The most observed category to **Vascular Plants** with 4470 observations whereas the second most observed is **Birds**. Therefore, due to this type of visualization the rest of the categories have low visibility in the plot as there are much less observations in comparison to **Vascular Plants**.

In [ ]:
# Set table for protected species. Proportion of protected species per category
protected_species = species.groupby('category')['is_protected'].mean().reset_index()
protected_species.columns = ['category', 'prop_species']
protected_species['prop_species'] = protected_species['prop_species']*100
protected_species = protected_species.sort_values(by = ['prop_species'], ascending = True)

# Create the barplot of the proportion of protected species
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data = protected_species, x = 'prop_species', y = 'category', color = PALETTE['accent'], ax=ax)
ax.set_title('Rate of Protected Species Per Category')
ax.set_xlabel('Rate Protected Species(%)')
ax.set_ylabel('Category')
ax.set_xlim(xmin = 0, xmax = 19)
for bar in ax.patches:
    width = bar.get_width()
    ax.text(width + 0.2, bar.get_y() + bar.get_height()/2,
            f'{width:.1f}%', va='center', fontsize=9,
            color='#E0E0E0')
plt.tight_layout()


# Saves the image in the images folder
plt.savefig('images/Rate of Protected Species per Category.png', dpi=150, bbox_inches='tight')

plt.show()
plt.close()

In [ ]:
# Table of number of observations of each category per level of conservation status
status_by_category = species.groupby(
    ['category', 'conservation_status_clean']
).size().unstack(fill_value=0)
print(status_by_category)

In [ ]:
# Set table without the 'No Concern' values. Only focusing on the protected species
status_pct = status_by_category.div(status_by_category.sum(axis=1), axis = 0)*100
# Not accounting for 'No Concern'
status_order_plot = ['Endangered', 'Threatened', 'Species of Concern', 'In Recovery']
# New table, copy of the status_pct to not change that data
risk_sorted = status_pct[status_order_plot].copy()
# Creates total risk column for to sum all 'Concern' species
risk_sorted['total_risk'] = risk_sorted.sum(axis=1)
# Sort by level of total risk
risk_sorted = risk_sorted.sort_values('total_risk', ascending=False)
risk_sorted = risk_sorted.drop(columns = 'total_risk')
#.drop(columns='total_risk')
print(risk_sorted.round(1))

In [ ]:
# Heatmap for better visualization of level of danger vs category
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    risk_sorted,
    annot=True,
    fmt='.1f',
    ax=ax,
    cmap='YlOrRd',
    linewidths = .5,
    linecolor='#12121C'
)

ax.set_title('Conservation Risk Profile by Taxonomic Category (% of species per category)')
ax.set_xlabel('Conservation Status')
ax.set_ylabel('Category')

plt.tight_layout()

# Saves the image in the images folder
plt.savefig('images/Conservation Risk Profile Heatmap.png', dpi=150, bbox_inches='tight')

plt.show()
plt.close()

### Key Findings

The conservation risk profile differs substantially across taxonomic categories:

- **Mammals and Birds** carry the highest overall conservation burden, 
  with **17.8%** and **15.2%** of their species holding an active 
  conservation status — roughly 10× more than either plant category.

- Prioritizing by **Endangered** status: Mammals (3.3%), Fish (2.4%), 
  and Amphibians (1.2%) are the most critical groups. Reptiles and 
  plants show zero endangered species in this dataset.

- **Birds** lead in *Species of Concern* (13.8%), followed closely by 
  Mammals (13.1%). This suggests widespread low-level concern rather 
  than acute crisis — but the volume is significant.

- **Fish** stand out for their *Threatened* rate (3.1%) relative to 
  their small total population (127 species), indicating disproportionate 
  vulnerability in this group.

- Both plant categories (**Vascular** and **Nonvascular**) show minimal 
  conservation risk — under 2%, exclusively at the *Species of Concern* 
  level. This likely reflects monitoring gaps as much as actual safety.

## 3. Statistical Significance Testing

In [ ]:
# To check if there is a relation between the categories and their level of protection
# or the level of protection is independent from the category
contingency_table = pd.crosstab(species.category, species.is_protected)
print(contingency_table)
print()
# To check if there is a significant relation we estimate a significance threshold of 0.05
significance_threshold = 0.05
# Chi Square Contingency for these two categorical variables
chi2, pval, dof, expected = chi2_contingency(contingency_table)
# Values obtained
print(f"Chi²  = {chi2:.4f}")
print(f"p-value = {pval:.2e}")
print(f"Degrees of freedom = {dof}")
print()
if pval < significance_threshold:
    print("Result: SIGNIFICANT — category and protection status are NOT independent.")
else:
    print("Result: NOT significant — no evidence of dependence.")

In [ ]:
# Cramér's V size of the effect V = sqrt(Chi2/(n*(k-1))
# n = sum of all values in contingency table
# k = min number of rows or columns
# V = Cramer's V (cramers_v)
n = contingency_table.sum().sum()
k = min(contingency_table.shape)
cramers_v = np.sqrt(chi2/(n*(k-1)))
print(f"Cramér's V = {cramers_v:.2f}")

if cramers_v < 0.1:
    print('Negligible Effect')
elif cramers_v< 0.3:
    print('Weak Effect')
elif cramers_v< 0.5:
    print('Moderate Effect')
else:
    print('Strong Effect')

### Key Findings

The taxonomic category and the protection status of the species are not independent:

- The Chi-square test yields **χ² = (469.51), p = 3.10e-98**, far below the **significance threshold of 0.05**, confirming that taxonomic category and conservation status are not independent.

- The obtained value of **Cramér's V = 0.28** indicates that there is a **weak** relation between the taxonomic categories of the species and their protection status. This suggests that the **taxonomic** category, although dependent **on** the status of protection, is not a strong predictor for determining the protection status of the species.

- **Possibly**, there are other factors outside the scope of our analysis that are worth considering, such as:
  - Habitat size
  - Species' visibility
  - Politics
  - Funding

- Furthermore, **the** categories that are in the worst protection status are **mammals and birds**. Such a worse status could be given **because** the **ecosystems** that suffer more from human activity are the ones that **mammals and birds** need the most for their survival.

- To **conclude**, it is worth considering the high level of observations of the **plant kingdom** with **respect to** the **animal kingdom** and how the first category does not tend to end up in critical levels of protection status. Such an observation could be due to a phenomenon known as **plant blindness**. Humans tend to ignore plants as species at risk while being the base of practically every **ecosystem**.

## 4. Species Observations Across Parks

In [ ]:
# Check if there will be data loss between the tables
obs_species = set(observations['scientific_name'].unique())
sp_species = set(species['scientific_name'].unique())

only_in_obs = obs_species - sp_species       
only_in_species = sp_species - obs_species
# if one of them gives 0, there is no data loss. If both give 0 we can savely use inner join
print(f"Only in observations: {len(only_in_obs)}")      # Left join
print(f"Only in species_info: {len(only_in_species)}")  # Right join

In [ ]:
df_complete = observations.merge(species, on='scientific_name', how='inner')
df_complete.head()

In [ ]:
# Total activity per park
df_act_park = df_complete[['park_name', 'observations']]
df_act_park = df_act_park.groupby('park_name').sum().reset_index()
df_act_park = df_act_park.sort_values(by = 'observations',ascending = True)
print(df_act_park)

fig, ax = plt.subplots(figsize = (10,5))
sns.barplot(data = df_act_park, x = 'observations', y = 'park_name', color = PALETTE['primary'], ax=ax)
ax.set_title('Number Of Observations Per Park')
ax.set_xlabel('Observations')
ax.set_ylabel('Parks')
for bar in ax.patches:
    width = bar.get_width()
    ax.text(width + 15000, bar.get_y() + bar.get_height()/2,
            f'{width:.2e}', va='center', fontsize=9,
            color='#E0E0E0')
ax.set_xlim(xmin = 0, xmax=1.75e6)

# Saves the image in the images folder
plt.savefig('images/Number of Observations per Park.png', dpi=150, bbox_inches='tight')

plt.show()
plt.close()

In [ ]:
# 10 most observed species
df_obs_spe = df_complete[['scientific_name', 'observations']]
df_obs_spe = df_obs_spe.groupby('scientific_name').sum().reset_index()
df_obs_spe = df_obs_spe.sort_values(by = 'observations',ascending = False).iloc[:10]
df_obs_spe = df_obs_spe.sort_values(by = 'observations',ascending = True)
print(df_obs_spe)

fig, ax = plt.subplots(figsize = (10,5))
sns.barplot(data = df_obs_spe, x = 'observations', y = 'scientific_name', color = PALETTE['warn'], ax=ax)
ax.set_title('10 Most Observed Species')
ax.set_xlabel('Observations')
ax.set_ylabel('Species')
for bar in ax.patches:
    width = bar.get_width()
    ax.text(width + 40, bar.get_y() + bar.get_height()/2,
            f'{width:.0f}', va='center', fontsize=9,
            color='#E0E0E0')
ax.set_xlim(xmin = 0, xmax=5650)

# Saves the image in the images folder
plt.savefig('images/10 Most Observed Species.png', dpi=150, bbox_inches='tight')

plt.show()
plt.close()

In [ ]:
# Observations by taxonomic category group
df_obs_cat = df_complete[['category', 'observations']]
df_obs_cat = df_obs_cat.groupby('category').sum().reset_index()
df_obs_cat = df_obs_cat.sort_values(by = 'observations',ascending = True)
print(df_obs_cat)

fig, ax = plt.subplots(figsize = (10,5))
sns.barplot(data = df_obs_cat, x = 'observations', y = 'category', color = PALETTE['accent'], ax=ax)
ax.set_title('Observations per Taxonomic Category')
ax.set_xlabel('Observations')
ax.set_ylabel('Category')
for bar in ax.patches:
    width = bar.get_width()
    ax.text(width + 17000, bar.get_y() + bar.get_height()/2,
            f'{width:.2e}', va='center', fontsize=9,
            color='#E0E0E0')
ax.set_xlim(xmin = 0, xmax=3.05e6)


# Saves the image in the images folder
plt.savefig('images/Observations per Taxonomic Category.png', dpi=150, bbox_inches='tight')

plt.show()
plt.close()

### Key Findings

- The Park with the highest number of observations is the **Yellowstone National Park** with **1.59 Millon** observations.

- The most observed species were:
    - **Streptopelia decaocto** (Eurasian Collared-Dove) with **5355** observations.
    - **Holcus lanatus** (Common Velvet Grass) with **5340** observations.
    
- The eight most observed species fall in the range between **4600 - 5300** observations. The following most observed species present half of that quantity less than **2600** observations.

- The order of the number of observations per Taxonomic Category is very similar for one for number of species per category. The only difference is that **Amphibians** have a lower number of observations and a bigger number of species than **Reptiles**

## 5. Conclusions & Recommendations

### Conclusions

- From the total amount of **5,824 species observed, 96.7% have no active conservation concern** flagged by the NPS. Regarding the other **191 species with an active status**, the majority of species fall into the **Species of Concern** conservation status. The taxonomic groups with the highest overall conservation status are **Mammals and Birds**, with **17.8%** and **15.2%** of their respective species carrying an active conservation status, respectively.

- The Chi² test revealed that the protection status of the species and the taxonomic categories are not independent. The values obtained (**χ² = 469.51, p-value = 3.10 × 10⁻⁹⁸**) for a **significance threshold of 0.05 (our p-value is well below)** indicate a significant relationship between both features. Furthermore, the calculated value of **Cramér's V = 0.28** also indicates a **weak** relationship between these features. Thus, suggesting that the taxonomic category cannot be considered a strong predictor of the protection status of the species.

- **Yellowstone National Park**, with **1.59 million** observations, represents the National Park in the study with the highest number of observations; this is **43.55% of the overall observations from all National Parks**. Both **Streptopelia decaocto** (Eurasian Collared-Dove) and **Holcus lanatus** (Common Velvet Grass), with **5,355** and **5,340** observations, respectively, are the most observed species. Six more species fall in the range between **4,600 and 5,300** observations. The rest of the species present half or less of that quantity.

- The order of the number of observations per taxonomic category is very similar to the number of species per category. That is, from most to least observed: **Vascular Plants, Birds, Non-Vascular Plants, Mammals, Fish, Amphibians, and Reptiles**. The only difference is that **Amphibians** have a lower number of observations and a larger number of species than **Reptiles**.

- Although **Vascular Plants** present the highest number of species observed, they only represent **1% of the species with an active protection status**. Such a difference could represent a monitoring gap following the **Plant Blindness Phenomenon**. From the 10 most observed species, **4 are Mammals, 4 are Vascular Plants, and 2 are Birds**. The fact that such a high number of **Mammals and Birds** are in the top ten of most observed species could indicate a higher interest in observing those taxonomic categories compared to others.


### Recommendations

- There is a possibility that the small number of **Plants** with an active protection status—despite being the most observed taxonomic category—could represent monitoring gaps rather than actual 'safe' species. Cross-validate 'NaN' values in conservation_status against official NPS records to confirm whether absence of status genuinely indicates no concern.

- Include the habitat sizes of the species and a factor that represents the species' visibility. There may be fewer observations of some species simply because the studied parks do not have a habitat for them to exist or because some species tend to hide more than others.

- Investigate the habitats that humans tend to destroy the most and observe the percentage of taxonomic categories that are more affected by them.


### Limitations

- There may be other factors outside our data that could affect the level of observations of species, such as: habitat size, species' visibility, politics, and funding.

- Our data does not specifically indicate that the unspecified values in the conservation status feature of the database actually represent that those species are 'safe'.

- Our data comes from 4 National Parks in the USA. In order to conduct a global analysis, we would need to include National Parks from all over the globe. Moreover, as a study focused only on the USA, there should be a larger number of National Parks, spread across the country, to ensure that at least the majority of the ecosystems in the country are included in the data.
- The observation data covers only a **7-day window**, which may not be representative of long-term species presence.

Despite these limitations, this analysis provides a statistically grounded overview of conservation priorities across taxonomic groups in U.S. national parks. The findings suggest that conservation efforts are not uniformly distributed — and that data gaps, particularly around plant species, may be obscuring a more complex picture of biodiversity risk.